In [1]:
import vcf
import sys
import argparse
import numpy as np
import pandas as pd
import math

<h2>Maps for sex ratio demography analysis</h2>

In [7]:
g1 = 1.5
g2 = 2
d1 = 590000
d2 = 820000
d3 = 1090000
res = []
df1 = pd.read_csv(f"/Volumes/WD/chimerism/marmoset_divergence/gaps/kuhlii/100kb_divergence.bed", sep='\t', names=['chrom','start','end','length','point_muts', 'divergence'])
df1 = df1[df1.chrom=="NC_071464.1"]
df1 = df1[df1.length>1000]
df1['mu'] = df1.divergence / (d2/g2)
#Set mutation bias
mb = 2.7
#Get male and female rates based on mu = ((2*mu_f) / 3) + ((mb * mu_f) /3)
df1['mu_f'] = (3* df1.mu) / (2+mb)
df1['mu_m'] = df1.mu_f * 2.7
df1[['mu', 'mu_f', 'mu_m']].describe()

,mu,mu_f,mu_m
count,3.940000e+02,3.940000e+02,3.940000e+02
mean,2.291929e-09,1.462933e-09,3.949920e-09
std,2.695560e-09,1.720570e-09,4.645539e-09
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,6.229549e-10,3.976308e-10,1.073603e-09
50%,1.418863e-09,9.056575e-10,2.445275e-09
75%,2.796006e-09,1.784685e-09,4.818649e-09
max,1.549134e-08,9.888090e-09,2.669784e-08


In [8]:
#Set total number of windows and window sizes, and positions for 1mb simulated region
totalWins = 1000
windowSize = 1000
pos = [int(x+1) for x in range(0, totalWins*windowSize, windowSize)]

In [16]:
#Loop through replicates
for i in range(1, 101):
    #MU MAPS
    #Sample from empirical distribution
    samples = df1.mu_f.sample(1000, replace=True)
    #Create df and output to file
    df = pd.DataFrame([pos, samples]).T
    df.to_csv(f"/Volumes/WD/chimerism/marmoset_X/mu_f_maps/{i}.txt", sep='\t', header=False, index=False)
    #Repeat for males
    samples = df1.mu_m.sample(1000, replace=True)
    df = pd.DataFrame([pos, samples]).T
    df.to_csv(f"/Volumes/WD/chimerism/marmoset_X/mu_m_maps/{i}.txt", sep='\t', header=False, index=False)

    #REC MAPS
    np.random.seed(np.random.randint(1000000))
    m = 0
    while((m>1.01) | (m<0.99)):
        mu = np.random.uniform(0, 1*2, 1000)
        m = np.mean(mu)
    df = pd.DataFrame([pos, mu]).T
    df[1] = df[1] * 1e-8
    df.to_csv(f"/Volumes/WD/chimerism/marmoset_X/rr_maps/{i}.txt", sep='\t', header=False, index=False)

<h2>Segments and maps for genome scan null threshold simulations</h2>

In [15]:
df = pd.read_csv(r"/Volumes/WD/chimerism/marmoset_X/empirical_data/exons_only.merged.X2.bed", sep='\t', names=['chrom','start','end'])
#Remove PAR
df = df[df.start>1210000]

#Get gap from previous exon by subtracting start of current exon from end of previous one
lst = [1213836-1210000]
for i in range(1, len(df)):
        lst.append(df.iloc[i].start - df.iloc[i-1].end)
df['gap_from_previous_exon'] = lst
#divide in half to calculate right flank
df['left_flank'] = df.gap_from_previous_exon / 2

#Get gap from next exon by subtracting start of next exon from current one
lst = []
for i in range(0, len(df)-1):
        lst.append(df.iloc[i+1].start - df.iloc[i].end)
lst.append(146897247 - df.iloc[-1].end)
df['gap_from_next_exon'] = lst
#divide in half to calculate right flank
df['right_flank'] = df.gap_from_next_exon / 2

df = df[(df.left_flank>25000) & (df.right_flank>25000)].reset_index(drop=True)

#####Repeat above steps for this subset of exons#####
#Get gap from previous exon by subtracting start of current exon from end of previous one
lst = [1213836-1210000]
for i in range(1, len(df)):
        lst.append(df.iloc[i].start - df.iloc[i-1].end)
df['gap_from_previous_exon'] = lst
#divide in half to calculate right flank
df['left_flank'] = df.gap_from_previous_exon / 2

#Get gap from next exon by subtracting start of next exon from current one
lst = []
for i in range(0, len(df)-1):
        lst.append(df.iloc[i+1].start - df.iloc[i].end)
lst.append(146897247 - df.iloc[-1].end)
df['gap_from_next_exon'] = lst
#divide in half to calculate right flank
df['right_flank'] = df.gap_from_next_exon / 2

df['seg_start'] = np.ceil(df.start - df.left_flank)
df['seg_end'] = np.where((df.end + df.right_flank) % 1 == 0, (df.end + df.right_flank) + 1, np.floor(df.end + df.right_flank))
df.loc[0, 'seg_start'] = 1210000
df.loc[159, 'seg_end'] = 146897247

df['seg_length'] = df.seg_end - df.seg_start

#df[['chrom', 'seg_start', 'seg_end', 'seg_length']].to_csv(r"/Volumes/WD/chimerism/marmoset_X/empirical_data/segments_by_exon_X.bed", sep='\t', header=False, index=False)

In [16]:
rdf = pd.DataFrame()
for i in range(0, len(df)):
    exon = [int(df.index[i]+1) for x in range(1, 101)]
    starts = [int(df.iloc[i].seg_start) for x in range(1, 101)]
    ends = [int(df.iloc[i].seg_end) for x in range(1, 101)]
    lengths = [int(df.iloc[i].seg_length) for x in range(1, 101)]
    reps = [x for x in range(1, 101)]

    tdf = pd.DataFrame([exon, starts, ends, lengths, reps]).T
    rdf = pd.concat([rdf, tdf])

rdf = rdf.reset_index(drop=True)
rdf[5] = rdf.index
#rdf.to_csv(r"/Volumes/WD/chimerism/marmoset_X/sweep_scans/null_thresholds/X_null.params", sep='\t', header=False, index=False)

rdf['length'] = rdf[2] - rdf[1]

In [31]:
g1 = 1.5
g2 = 2
d1 = 590000
d2 = 820000
d3 = 1090000
res = []
df1 = pd.read_csv(f"/Volumes/WD/chimerism/marmoset_divergence/gaps/kuhlii/1kb_divergence.bed", sep='\t', names=['chrom','start','end','length','point_muts', 'divergence'])
df1 = df1[df1.chrom=="NC_071464.1"]
df1 = df1[df1.length>100]
df1['mu'] = df1.divergence / (d2/g2)
#Set mutation bias
mb = 2.7
#Get male and female rates based on mu = ((2*mu_f) / 3) + ((mb * mu_f) /3)
df1['mu_f'] = (3* df1.mu) / (2+mb)
df1['mu_m'] = df1.mu_f * 2.7

l = []
for exon in range(0, len(rdf)):
    totalWins = math.ceil(rdf.iloc[exon]['length']/1000)
    windowSize = 1000
    pos = [int(x+1) for x in range(rdf.iloc[exon][1], rdf.iloc[exon][2], windowSize)]

    mf=0
    while((mf>1.2e-9) | (mf<0.6e-9)):
        samples = df1.mu_f.sample(totalWins, replace=True)
        mf = samples.mean()
    print(str(mf))
    l.append(mf)
    df2 = pd.DataFrame([pos, samples]).T
    df2[0] = [int(x) for x in df2[0]]
    #df[1] = np.where(df[1]==0, df1[df1.mu>0].mu.min(), df[1])
    df2.to_csv(f"/Volumes/WD/chimerism/marmoset_X/sweep_scans/null_thresholds/mu_f_maps_null/{rdf.iloc[exon][0]}_rep{rdf.iloc[exon][4]}.txt", sep='\t', header=False, index=False)
    
    totalWins = math.ceil(rdf.iloc[exon]['length']/1000)
    windowSize = 1000
    pos = [int(x+1) for x in range(rdf.iloc[exon][1], rdf.iloc[exon][2], windowSize)]

    mf=0
    while((mf>3e-9) | (mf<1.5e-9)):
        samples = df1.mu_m.sample(totalWins, replace=True)
        mf = samples.mean()
    print(str(mf))
    l.append(mf)
    df2 = pd.DataFrame([pos, samples]).T
    df2[0] = [int(x) for x in df2[0]]
    #df[1] = np.where(df[1]==0, df1[df1.mu>0].mu.min(), df[1])
    df2.to_csv(f"/Volumes/WD/chimerism/marmoset_X/sweep_scans/null_thresholds/mu_m_maps_null/{rdf.iloc[exon][0]}_rep{rdf.iloc[exon][4]}.txt", sep='\t', header=False, index=False)

1.0314362849031671e-09
2.93368764980616e-09
9.896661532490527e-10
2.7926833362260318e-09
1.0283207633234923e-09
2.9633259407011378e-09
1.1109479218360932e-09
2.717579314474103e-09
1.0640566247879534e-09
2.7619349714146168e-09
9.895272016850332e-10
2.755268256923942e-09
9.93812915098577e-10
2.7969875539546383e-09
1.1348344088647394e-09
2.5652666855346186e-09
9.846313745840836e-10
2.9228179294675837e-09
1.107534677929293e-09
2.395788695887281e-09
1.0749354498000547e-09
2.851446290092581e-09
1.0610512086100659e-09
2.7649478004299802e-09
9.195044136180678e-10
2.613536548298614e-09
1.0247412052522099e-09
2.662891245590089e-09
1.0230410460811205e-09
2.5161160486322186e-09
1.0274754479684972e-09
2.7770048623497144e-09
1.0713555550710164e-09
2.993275129626319e-09
1.0167040603890753e-09
2.6022876083780684e-09
1.044626755569898e-09
2.772915233589025e-09
1.042686976316214e-09
2.9332171487879015e-09
1.0837477735671332e-09
2.833893646762286e-09
1.0321091006615412e-09
2.932417549026876e-09
1.1354010

In [93]:
mu = 0.229e-8
theta = 0.0003
Ne= theta / (3 * mu)
lst = []

for i in range(1, 11):
    df = pd.read_csv("/Volumes/WD/chimerism/marmoset_X/rec_sims/LDhat/results/rep"+str(i)+"_res.txt", sep='\t', header=0).iloc[1:]
    # LDhat estimates in 4Ne r / kb. Therefore, multiple by 1000 to get units in Mb, and by another 100 to go from M to cM, then divide by 4Ne
    df['r'] = (df.Mean_rho / 1000) /(3*Ne)
    lst.append(df.r.mean())

lh = np.mean(lst)

mu = 0.229e-8
theta = 0.0005
Ne= theta / (3 * mu)

df = pd.read_csv(r"/Volumes/WD/chimerism/marmoset_X/empirical_data/rec/LDhat/10kb_rho.bed", sep='\t', names=['chrom', 'start', 'end', 'rho'])
# LDhat estimates in 4Ne r / kb. Therefore, multiple by 1000 to get units in Mb, and by another 100 to go from M to cM, then divide by 4Ne
df = df[df.start>1210000]
df['r'] = (df.rho / 1000) /(3*Ne)
#df['r'] = df.r * (2/3)
df['r'] = df['r'] / (lh/1e-8)

l = []
for exon in range(15900, len(rdf)):
    totalWins = math.ceil(rdf.iloc[exon]['length']/1000)
    windowSize = 1000
    pos = [int(x+1) for x in range(rdf.iloc[exon][1], rdf.iloc[exon][2], windowSize)]

    mf=0
    while((mf>8.5e-10) | (mf<5e-10)):
        samples = df.r.sample(totalWins, replace=True)
        mf = samples.mean()
    print(str(mf))
    l.append(mf)
    df2 = pd.DataFrame([pos, samples]).T
    df2[0] = [int(x) for x in df2[0]]
    #df[1] = np.where(df[1]==0, df1[df1.mu>0].mu.min(), df[1])
    df2.to_csv(f"/Volumes/WD/chimerism/marmoset_X/sweep_scans/null_thresholds/rr_maps_null/{rdf.iloc[exon][0]}_rep{rdf.iloc[exon][4]}.txt", sep='\t', header=False, index=False)

8.359333224353538e-10
8.401694491527362e-10
8.459840486111055e-10
8.411896305002576e-10
8.459799227323997e-10
8.310642731350616e-10
8.411878591059864e-10
8.436327711879109e-10
8.396116968002409e-10
8.252078058190358e-10
8.485643845130051e-10
8.361367814777821e-10
8.396303165869812e-10
8.386951430429217e-10
8.4193817871781e-10
8.42298113481054e-10
8.475515571345577e-10
8.325441651134471e-10
8.278752296699671e-10
8.442425692613248e-10
8.27816516835877e-10
8.352050775259649e-10
8.408980790913494e-10
8.499664791390224e-10
8.28171932346637e-10
8.331660648354751e-10
8.406660307833013e-10
8.413291715143912e-10
8.402951833384726e-10
8.422684648361244e-10
8.472610057698104e-10
8.309486725870634e-10
8.454225982590312e-10
8.136822353741616e-10
8.474723037940504e-10
8.462018733406586e-10
8.436039783364841e-10
8.394152269953991e-10
8.494590001046676e-10
8.449574459446246e-10
8.481915890140424e-10
8.337764539384872e-10
8.292775778329869e-10
8.439167242608898e-10
8.457026947204562e-10
8.4668442211558